# Почему детектор хуже работает ночью

Разбор к докладу. Все числа считаются из `boxes.csv`, который делает
`analyze_contrast.py` — здесь только отрисовка, логика лежит в `src/analysis/`.

**Краткий вывод.** Рабочая гипотеза «ночью объекты сливаются с тёмным фоном»
**не подтвердилась**. Ночью объекты выделяются на фоне даже сильнее, чем днём.
Настоящая причина ночного разрыва — **точность рамок**: модель находит объекты
почти так же хорошо, но обводит их заметно хуже.

### Как запустить на Kaggle

1. **File → Import Notebook → Upload** и выбрать этот `.ipynb`
   (в приватном репозитории ссылка на GitHub не сработает — только файл).
2. В **Add Input** подключить датасет `nvpdyf-bdd100k` и датасет с весами.
3. Ускоритель **не нужен**: здесь только чтение таблицы и отрисовка.
4. В ячейке ниже раскомментировать `git clone` — из репозитория берётся
   `src/analysis/`, вся логика лежит там.
5. Если `boxes.csv` ещё не посчитан, сначала прогнать `analyze_contrast.py`
   (см. `docs/KAGGLE.md`) — он и создаёт эту таблицу.

In [ ]:
import sys
from pathlib import Path

# --- на Kaggle: склонировать репозиторий (раскомментируй один раз) ---
# !git clone -q https://github.com/Antonoof/Yandex-night-vision-Detection.git /kaggle/working/repo

# Корень репозитория. На Kaggle это путь клона, локально — папка над notebooks/.
REPO = Path('/kaggle/working/repo')
if not (REPO / 'src').is_dir():
    REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print('репозиторий:', REPO, '| src найден:', (REPO / 'src').is_dir())

import logging
logging.basicConfig(level=logging.INFO, format='%(message)s', force=True)

from IPython.display import Image, display
from src.analysis import (
    contrast_control_summary, draw_localization_grid,
    localization_decomposition, localization_summary,
    plot_decomposition, plot_localization,
    plot_overview, plot_run_comparison, read_boxes_csv, recall_by_contrast,
    summarize,
)

# ---- пути к данным: поправь под своё окружение ----
ANALYSIS_DIR = Path('/kaggle/working/repo/saved/analysis/contrast')  # где boxes.csv
IMAGES_DIR   = Path('/kaggle/input/nvpdyf-bdd100k/images/val')       # кадры val
OUT = REPO / 'saved' / 'analysis' / 'figures'
OUT.mkdir(parents=True, exist_ok=True)
print('boxes.csv на месте:', (ANALYSIS_DIR / 'boxes.csv').exists())

## 1. Где мы находимся

Четыре сопоставимых прогона на одном датасете (7 классов, `nvpdyf`).
Аугментации дали +0.03, Zero-DCE — ноль, разрешение — больше всех.
**Разрыв ночь/день при этом не сдвинулся ни от одного из них.**

In [ ]:
runs = [
    {'label': '№8  augment=none',   'night': 0.2153, 'day': 0.2420},
    {'label': '№11 +аугментации',   'night': 0.2454, 'day': 0.2749},
    {'label': '№13 +Zero-DCE',      'night': 0.2444, 'day': 0.2717},
    {'label': '№14 imgsz=960',      'night': 0.2882, 'day': 0.3302},
]
display(Image(str(plot_run_comparison(runs, OUT / 'runs.png'))))

## 2. Гипотеза: «объекты сливаются с тёмным фоном»

Проверяем тремя независимыми измерениями по каждому размеченному боксу:
контраст Вебера против кольца фона, динамический диапазон яркости внутри
бокса и состав выборки по размеру.

In [ ]:
rows = read_boxes_csv(ANALYSIS_DIR / 'boxes.csv')
summarize(rows)

Читать так:

* **контраст Вебера: ночь +0.30, день −0.07** — ночью объект *ярче* фона
  (его подсвечивают фары, или он сам светится), днём он чуть темнее
  равномерно освещённого окружения. Ночью объекты выделяются **сильнее**;
* **динамический диапазон: 124 против 127** — одинаков. Объектов, у которых
  на всё про всё меньше 10 уровней яркости, ночью 0.3%. Тональный диапазон
  занят полностью, растягивать было нечего — вот и объяснение, почему
  Zero-DCE не дал ничего;
* **состав по размеру идентичен** — small 50.2% ночью против 50.1% днём.
  Разрыв не объясняется и тем, что «ночью объекты мельче».

Гипотеза в исходной формулировке опровергнута.

In [ ]:
curves = recall_by_contrast(rows)
display(Image(str(plot_overview(rows, OUT / 'contrast.png', recall_curves=curves))))

## 3. Настоящая причина — точность рамок

Раз объекты находятся, а mAP ниже, разница должна быть в качестве рамок.
Смотрим IoU **среди уже найденных** объектов.

In [ ]:
localization_summary(rows)

In [ ]:
display(Image(str(plot_localization(rows, OUT / 'localization.png'))))

Правый нижний график — главный. Чем строже требование к точности рамки,
тем сильнее ночь отстаёт: с 6.8% при IoU≥0.5 до 47% при IoU≥0.95.

Если бы проблема была в *обнаружении*, обе кривые сдвинулись бы вниз на
константу. Они расходятся — значит дело в *локализации*.

То же самое видно в журнале, независимо от этого скрипта: разрыв по
`mAP@50` — **7.92%**, по `mAP@50-95` — **10.73%**.

## 4. Как именно рамка неправильная

IoU говорит, что рамка плохая, но не говорит, чем. Разложим ошибку каждого
бокса на две части: **смещение** центра и **ошибку размера**, обе — в долях
размера объекта (иначе статистика просто померила бы, какие объекты крупнее).

Вопрос содержательный, а не косметический. *Систематическая* ошибка и
*случайная* лечатся по-разному: если ночью модель раздувает рамки по ореолу
фар — это смещение, и его можно откалибровать; если рамка просто дрожит —
калибровать нечего, нужен другой сигнал на границе.


In [ ]:
localization_decomposition(rows)

Медианы ночью и днём совпадают — и по центру, и по отношению площадей
(1.022 против 1.020). **Систематического смещения нет:** модель ночью не
раздувает боксы по свечению и не жмёт их к освещённой части. В среднем она
права.

А разброс шире на 26–35% по каждой из четырёх координат. Ночная ошибка
локализации — **дисперсионная**.

Два разреза стоит проговорить отдельно:

* **разрыв растёт с размером объекта** (dx: +3% на small, +37% на medium,
  +43% на large). Контринтуитивно: главный рычаг проекта — мелкие объекты, а
  по относительной точности границ ночь сильнее всего проседает на крупных.
  Абсолютные величины там крошечные (0.0203 против 0.0142) — но именно там
  IoU 0.9+ достижим днём и недостижим ночью, а `mAP@50-95` считает пороги
  вплоть до 0.95;
* **вертикаль хуже горизонтали.** У `traffic light` разброс по вертикали
  +53%, а по горизонтали **−5%**, то есть ночью не хуже вообще. Читается так:
  ночью объект очерчен собственными огнями, они задают левую и правую
  границы, а верх и низ — крыша, тень под машиной, корпус светофора — тонут
  в темноте.


### Контроль: а не в контрасте ли всё-таки дело

Последний оставшийся конфаундер: вдруг ночные боксы дрожат просто потому, что
ночные объекты живут при других контрастах. Фиксируем класс, размер **и**
контраст одновременно и смотрим, остаётся ли что-нибудь.


In [ ]:
contrast_control_summary(rows)

In [ ]:
display(Image(str(plot_decomposition(rows, OUT / 'decomposition.png'))))

Остаётся: при равном классе, размере и контрасте ночь всё равно хуже на
30–52%. Тот же вывод, что и в разделе 2, но теперь он держится на уровне
границ, а не только обнаружения.

**Оговорка, которую нельзя опускать.** Мы меряем расхождение предсказания и
разметки, а не ошибку модели. Если ночью разметчик тоже видит только
освещённые части объекта, дисперсия самой разметки ночью выше — и в наши
числа она входит неотличимо. Отсюда практическое следствие: при шумных
метках повышенный вес `box` заставляет модель точнее подгонять шум, так что
нулевой результат прогона №30 будет иметь **два** объяснения, а не одно.


## 5. То же самое глазами

Почему мы не замечали этого в Comet: там разметка и предсказания нарисованы
в разных панелях на полном кадре 1280×720. В таком масштабе рамка, съехавшая
на двадцать пикселей, выглядит правильной.

Здесь оба бокса наложены на один кроп вокруг объекта: **сплошная зелёная —
разметка, пунктирная красная — предсказание**. Выборка не подтасована — это
объекты из типичной для ночи полосы IoU 0.4–0.8.

In [ ]:
for tod, title in (('night', 'ночь'), ('daytime', 'день')):
    path = draw_localization_grid(
        rows, IMAGES_DIR, OUT / f'boxes_{tod}.png',
        timeofday=tod, n=8, iou_range=(0.4, 0.8),
    )
    if path:
        display(Image(str(path)))

### Кто ошибся — модель или разметка?

Раздел 4 закончился оговоркой: расхождение предсказания и разметки — это не
обязательно ошибка модели. Единственный доступный нам способ проверить —
посмотреть на худшие случаи глазами и честно назвать виновного.

Полоса IoU здесь другая, 0.2–0.5: это уже не типичный случай, а те кадры, где
расхождение велико настолько, что видно, кто прав. В подписи — самое уехавшее
ребро в пикселях.


In [ ]:
display(Image(str(draw_localization_grid(
    rows, IMAGES_DIR, OUT / 'boxes_night_worst.png',
    timeofday='night', n=12, ncols=4, iou_range=(0.2, 0.5), min_area=4000, require_matched=False,
))))

Считать глазами по этой сетке: у скольких из 12 разметка обводит объект
целиком, а предсказание — только освещённую часть (тогда виновата модель), и
у скольких наоборот, разметка сама обрывается по краю света (тогда часть
нашего «разрыва локализации» — шум меток, и поднимать вес `box` бессмысленно).

Это не измерение, а зрячая проверка на дюжине примеров — так и надо
произносить на докладе. Настоящий ответ дала бы только переразметка.


## 6. Что из этого следует

1. **Ночной разрыв — это разрыв в локализации, а не в детекции.** Модель
   видит объекты почти так же хорошо, но обводит их хуже: медиана IoU 0.763
   против 0.812, доля IoU>0.9 — 17.2% против 25.7%.
2. **Zero-DCE решал несуществующую проблему.** Тональный диапазон ночных
   объектов и так полный; тонмаппингу нечего было растягивать.
3. **Разрешение помогло больше всего именно потому**, что даёт то, чего не
   хватало, — пиксели на границу объекта: `imgsz=960` почти удвоил
   `map_small` ночью (0.0477 → 0.0860).
4. **Ошибка ночью — дисперсионная, а не систематическая.** Рамка не смещена
   и не раздута — она дрожит: разброс шире на 26–35% при совпадающих
   медианах. Калибровать нечего, нужен более надёжный сигнал на границе.
5. **Хуже всего верхнее и нижнее рёбра** (+39% и +30% против +27% и +23% по
   бокам). Это меняет план аугментации «день→ночь»: смаз от движения камеры
   горизонтальный, он портил бы вертикальные рёбра, а у нас наоборот. Похоже,
   ночь съедает не резкость, а **неосвещённые части объекта**, — и тогда
   имитировать надо не затемнение кадра и не блюр, а потерю контраста там,
   где нет источника света, плюс свечение вокруг ярких пикселей.

### Оговорки

* recall считался при `conf=0.25` — это «нашёл ли детектор объект на
  человеческом пороге», а не mAP; в журнал эти числа не переносятся;
* у `bicycle`, `bus`, `motorcycle` ночью 48–149 боксов, их «ночь лучше дня» —
  статистический шум;
* `dyn_range` меряется внутри прямоугольного бокса, куда попадают фон и фары,
  поэтому у мелких ночных объектов он завышен (148 против 118 днём): это
  яркие точки на тёмном, а не богатство деталей.